# 检查语言output能力

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
# NOTE enable this in single script
# current_file_path = os.path.dirname(os.path.abspath(__file__))
# module_path = os.path.join(current_file_path, "../..")
# sys.path.append(module_path)
from dataclasses import asdict
import math
from pathlib import Path
from typing import List, Optional
import yaml
import debugpy

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import transformers
from transformers import Trainer, is_datasets_available
import datasets
from transformers.integrations import deepspeed
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, RandomSampler, DataLoader

from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from models.qwen2_5_vl import Qwen2_5_VLRetForConditionalGeneration

from arguments import ModelArguments, DataArguments, TrainingArguments, LoraArguments
from collators import COLLATORS
from dataset.datasets_mbeir import LazySupervisedDataset, MbeirLanguageDataset
from dataset.datasets_xhs import XHSDataset
from dataset.datasets_dam import DAMDataset
# from dataset.datasets_mmeb import MMEBDataset
from loaders import LOADERS
from supported_models import MODULE_KEYWORDS
from utils import (
    rank0_print
)
    
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained("./checkpoints/Qwen2.5-VL-7B-Dam", attn_implementation= "flash_attention_2", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).cuda()
# model = Qwen2_5_VLRetForConditionalGeneration.from_pretrained("./checkpoints/Qwen2.5-VL-7B-Dam", attn_implementation= "flash_attention_2", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).cuda()
model = Qwen2_5_VLRetForConditionalGeneration.from_pretrained("./checkpoints/qwen2_5-vl-7b_DAM_pretrain_vision", attn_implementation= "flash_attention_2", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).cuda()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
# processor = AutoProcessor.from_pretrained("./checkpoints/qwen2_5-vl-7b_DAM_pretrain_vision")
tokenizer = processor.tokenizer 
tokenizer.padding_side  = 'left'


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[2025-07-08 12:05:57,620] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.5
 [WARNING]  using untested triton version (3.1.0), only 1.0.0 is known to be compatible


/usr/local/lib/python3.10/dist-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/usr/local/lib/python3.10/dist-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.92s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences

In [2]:

device = 'cuda:0'

def tensors_to_device(data, device, dtype=model.dtype):
    for key in data.keys():
        if isinstance(data[key], torch.Tensor):
            if key == 'pixel_values':
                data[key] = data[key].to(device).to(dtype)
            else:
                data[key] = data[key].to(device)
    return data 


model.to(device)
model.eval()
model.config.emb_token_ids = [-777]

In [11]:

from dataset.datasets_dam import DAMDataset
dam_dataset = DAMDataset(
    data_path="/mnt/tidalfs-hssh01/dataset/mmeb/describe-anything-data",
    max_length=10000,
    mode='crop', inference=True
)

data_collator = COLLATORS['qwen2_5-vl-7b'](
    tokenizer=tokenizer,
    processor=processor,
)

dataloader = DataLoader(dam_dataset, batch_size=1, num_workers=2, shuffle=False, collate_fn=data_collator)
dataloader = iter(dataloader)


# TODO : 跑之前要把dam ds返回的改成1
datalist = [d for d,i in zip(dataloader, range(10))]

with torch.no_grad():
    for i, data in enumerate(datalist):
        # data = next(dataloader)
        batch = tensors_to_device(data, device)
        output = model.generate(**batch, max_new_tokens=512)
        image_generated_ids = output
        image_input_length = batch['input_ids'].shape[1]
        image_generated_only_ids = image_generated_ids[:, image_input_length:]

        # 解码生成的 token IDs 为文本
        # `skip_special_tokens=True` 会移除像 <|endoftext|> 这样的特殊标记
        image_decoded_outputs = tokenizer.batch_decode(image_generated_ids, skip_special_tokens=True)
        print("@decode inputs:" ,image_decoded_outputs)

@decode inputs: ['system\nYou are a helpful assistant.\nuser\n\nDescribe the image in detail.\nassistant\nA rectangular window divided into four equal panes by two vertical and one horizontal wooden muntin. The window frame is dark brown, and the glass panes are clear with a slight reflection.']
@decode inputs: ['system\nYou are a helpful assistant.\nuser\n\nDescribe the image in detail.\nassistant\nThe river has a smooth, light gray surface with subtle ripples running horizontally across it. The water appears calm and reflective, with a slight gradient from a lighter shade at the top to a slightly darker shade towards the bottom.']
@decode inputs: ['system\nYou are a helpful assistant.\nuser\n\nDescribe the image in detail.\nassistant\nThe dress features a vibrant, multi-colored pattern with intricate designs and floral motifs. The bodice is adorned with a rich, golden fabric that contrasts beautifully with the colorful skirt. The skirt is layered, with a series of ruffled and pleated

In [20]:
i=-1
# model.visual.context_layers[i].mlp_factor
model.visual.context_layers[i].mlp_factor
model.visual.context_layers[i].norm1.weight

Parameter containing:
tensor([2.0625, 2.1875, 2.2344,  ..., 2.1875, 2.1875, 2.2188], device='cuda:0',
       dtype=torch.bfloat16, requires_grad=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pycocotools
from PIL import Image
import pickle

def counts_to_mask(maskrle):
    return np.array(pycocotools.mask.decode(maskrle), dtype=np.float32)

def visualize_mask_on_image_pil(original_pil, binary_mask_np, 
                                color=(255, 0, 0), alpha_percent=50):
    height, width = binary_mask_np.shape
    if original_pil.size != (width, height):
        print(f"Warning: Mask size ({width},{height}) and image size ({original_pil.size}) differ.")
    alpha_value = int((alpha_percent / 100.0) * 255)
    colored_mask_pil = Image.new("RGBA", original_pil.size, (0, 0, 0, 0))
    mask_rgba_np = np.zeros((height, width, 4), dtype=np.uint8)
    
    mask_indices = binary_mask_np == 1
    mask_rgba_np[mask_indices] = list(color) + [alpha_value]

    colored_mask_pil_from_np = Image.fromarray(mask_rgba_np, "RGBA")
    original_pil.putalpha(255)
    overlaid_image_pil = Image.alpha_composite(original_pil, colored_mask_pil_from_np)    
    return overlaid_image_pil

def mask2box(mask):
    box = None
    pos = np.where(mask > 0)
    height, width = mask.shape

    if pos[0].size > 0 and pos[1].size > 0:
        x_min = np.min(pos[1]) / width
        x_max = np.max(pos[1]) / width
        y_min = np.min(pos[0]) / height
        y_max = np.max(pos[0]) / height
        box = [x_min, y_min, x_max, y_max]
    return box


dataset = dam_dataset.dataset['SAM']['train']
idx = (12,0)
anno = pickle.loads(dataset[idx[0]]['pickle'])
print(anno[idx[1]])
mask = counts_to_mask(anno[idx[1]]['mask_rle'])
box = mask2box(mask)
# print(anno[idx[1]]['category'])
# print(anno[idx[1]]['caption'])
img = dataset[idx[0]]['jpg']
print(box)
width, height = img.size
x0 = width * box[0]
y0 = height * box[1]
x1 = width * box[2]
y1 = height * box[3]

img.crop((x0, y0, x1, y1)).show()
visualize_mask_on_image_pil(img, mask)

In [31]:
dataset[idx[0]]['jpg'].size

(1500, 2060)

# 测试xhs的描述

In [4]:
from dataset.datasets_xhs import XHSDataset
from dataset.datasets_mbeir import LazySupervisedDataset, MbeirLanguageDataset
mnt = "tidalfs-hssh01"
xhs_dataset = XHSDataset(
    query_data_path=f"/mnt/{mnt}/dataset/mmeb/M-BEIR/query/train/mbeir_xhsnote_task7_train.jsonl",
    cand_pool_path=f"/mnt/{mnt}/dataset/mmeb/M-BEIR/cand_pool/local/mbeir_xhsnote_task7_cand_pool.jsonl",
    instructions_path=f"/mnt/{mnt}/dataset/mmeb/M-BEIR/instructions/query_instructions.tsv",
    image_path_prefix=f"/mnt/{mnt}/dataset/M-BEIR",
    tokenizer=tokenizer 
)


In [16]:

data_collator = COLLATORS['qwen2_5-vl-7b'](
    tokenizer=tokenizer,
    processor=processor,
)

dataloader = DataLoader(xhs_dataset, batch_size=1, num_workers=0, shuffle=False, collate_fn=data_collator)
dataloader = iter(dataloader)


# TODO : 跑之前要把dataset_xhs的prompt改成描述类型的；吧mbeir_dataset中get random poscand改成固定的
datalist = [d for d,i in zip(dataloader, range(2))]

with torch.no_grad():
    for i, data in enumerate(datalist):
        # data = next(dataloader)
        batch = tensors_to_device(data, device)
        output = model.generate(**batch, max_new_tokens=1024)
        image_generated_ids = output
        image_input_length = batch['input_ids'].shape[1]
        image_generated_only_ids = image_generated_ids[:, image_input_length:]

        # 解码生成的 token IDs 为文本
        # `skip_special_tokens=True` 会移除像 <|endoftext|> 这样的特殊标记
        image_decoded_outputs = tokenizer.batch_decode(image_generated_ids, skip_special_tokens=True)
        print("@decode inputs:" ,image_decoded_outputs)

[([{'role': 'user', 'content': [{'type': 'image', 'image': '/mnt/tidalfs-hssh01/dataset/mmeb/xhs_data/note_data/20250304/images/1040g0083189pmk643k0g5n7rfq898st4v5d9jmg.jpg', 'box': [0.35006, 0.76561, 0.59137, 0.96301]}, {'type': 'text', 'text': '\nDescribe the image in detail.'}]}], [{'role': 'user', 'content': [{'type': 'image', 'image': '/mnt/tidalfs-hssh01/dataset/mmeb/xhs_data/note_data/20250304/images/1040g2sg313udgnsoh87g5oppbr9m5fp8hrmo6n8.jpg', 'box': [0.46301, 0.3773, 0.92572, 0.75576]}, {'type': 'text', 'text': '\nDescribe the image in detail.'}]}])]
[([{'role': 'user', 'content': [{'type': 'image', 'image': '/mnt/tidalfs-hssh01/dataset/mmeb/xhs_data/note_data/20250304/images/active_search_1040g0mg3181i0slljk1g49uktbgs606gp305c58.jpg', 'box': [0.3, 0.31, 0.9, 0.56]}, {'type': 'text', 'text': '\nDescribe the image in detail.'}]}], [{'role': 'user', 'content': [{'type': 'image', 'image': '/mnt/tidalfs-hssh01/dataset/mmeb/xhs_data/note_data/20250304/images/1040g2sg3133v93303810

@decode inputs: ['system\nYou are a helpful assistant.\nuser\n\nDescribe the image in detail.\nassistant\nA black, long-sleeved dress with a fitted bodice and a flared skirt. The dress features a high neckline and a zipper closure on the left side. The fabric appears to be smooth and slightly shiny, with a subtle texture that suggests a satin-like material. Th', 'system\nYou are a helpful assistant.\nuser\n\nDescribe the image in detail.\nassistant\nA black dress with a fitted bodice and a flared skirt. The bodice features a deep V-neckline and a high waistline, accentuated by a belt with a decorative buckle. The skirt is adorned with a pattern of white and silver metallic flowers and leaves, creating']


手动打开图片并且渲染路径

In [ ]:
xhs_dataset[0]

In [ ]:
from PIL import Image

Image.open(
'/mnt/tidalfs-hssh01/dataset/mmeb/xhs_data/note_data/20250304/images/1040g2sg3133v933038105n41s1blj9dmalkc2uo.jpg'
).show()